# LangGraph 核心設計思維

**課程：LLM Application - AI Agent｜Day 1 上午**

本筆記本涵蓋 LangGraph 的核心概念與三大 Agent 設計模式：ReAct、Plan-and-Execute、Supervisor。

In [ ]:
%%capture
!pip install langchain langchain-openai langgraph pydantic

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

---
## 1. 為什麼選 LangGraph？

### LangChain vs LangGraph

| 比較項目 | LangChain (LCEL) | LangGraph |
|---------|-----------------|----------|
| 執行模式 | 線性 Chain / 簡單分支 | 圖（Graph）結構，支援迴圈 |
| 狀態管理 | 隱式（透過 chain 傳遞） | 顯式（TypedDict / State） |
| 條件分支 | RunnableBranch（有限） | Conditional Edge（靈活） |
| 迴圈控制 | 不原生支援 | 原生支援（Agent Loop） |
| 適用場景 | 簡單的 prompt → LLM → parse | 多步驟 Agent、需要迴圈的工作流 |

### 什麼時候該用 Graph-based Orchestration？

- 需要 **迴圈**（Agent 反覆思考直到完成）
- 需要 **條件路由**（根據 LLM 輸出決定下一步）
- 需要 **顯式狀態管理**（多個節點共享、修改狀態）
- 需要 **多 Agent 協作**（Supervisor 調度多個 Worker）

---
## 2. LangGraph 核心概念

LangGraph 的四個核心元件：

- **StateGraph**：定義整個圖的結構，綁定狀態型別
- **Node**：圖中的節點，每個節點是一個函數，接收 state、回傳 state 更新
- **Edge**：節點之間的固定連線
- **Conditional Edge**：根據條件動態決定下一個節點

In [ ]:
# --- Hello World Graph ---
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 定義狀態
class HelloState(TypedDict):
    message: str

# 定義節點函數（flat function，不包裝 class）
def say_hello(state: HelloState) -> dict:
    """打招呼節點"""
    return {"message": "Hello, LangGraph!"}

def add_emoji(state: HelloState) -> dict:
    """加上 emoji 節點"""
    return {"message": state["message"] + " 🚀"}

# 建立 Graph
graph = StateGraph(HelloState)

# 加入節點
graph.add_node("say_hello", say_hello)
graph.add_node("add_emoji", add_emoji)

# 加入邊：START → say_hello → add_emoji → END
graph.add_edge(START, "say_hello")
graph.add_edge("say_hello", "add_emoji")
graph.add_edge("add_emoji", END)

# 編譯並執行
app = graph.compile()
result = app.invoke({"message": ""})
print("結果:", result["message"])

---
## 3. Agent 狀態管理 (State Management)

LangGraph 使用 **TypedDict** 定義狀態結構，每個節點：
- 接收完整的 `state`
- 回傳要更新的部分（字典）
- LangGraph 自動 merge 回狀態

以下我們建一個 3 節點 pipeline：**input → process → output**

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 定義狀態：包含原始輸入、處理結果、最終輸出
class PipelineState(TypedDict):
    raw_input: str       # 原始輸入
    processed: str       # 處理後的資料
    final_output: str    # 最終輸出

# 節點 1：接收輸入
def input_node(state: PipelineState) -> dict:
    print(f"[input_node] 收到原始輸入: {state['raw_input']}")
    return {"raw_input": state["raw_input"].strip()}

# 節點 2：處理資料
def process_node(state: PipelineState) -> dict:
    processed = state["raw_input"].upper()  # 簡單處理：轉大寫
    print(f"[process_node] 處理後: {processed}")
    return {"processed": processed}

# 節點 3：產出結果
def output_node(state: PipelineState) -> dict:
    result = f"最終結果: {state['processed']} (長度: {len(state['processed'])})"
    print(f"[output_node] {result}")
    return {"final_output": result}

# 建立 Graph
pipeline = StateGraph(PipelineState)
pipeline.add_node("input", input_node)
pipeline.add_node("process", process_node)
pipeline.add_node("output", output_node)

# 連接邊
pipeline.add_edge(START, "input")
pipeline.add_edge("input", "process")
pipeline.add_edge("process", "output")
pipeline.add_edge("output", END)

# 編譯並執行
app = pipeline.compile()
result = app.invoke({
    "raw_input": "  hello langgraph  ",
    "processed": "",
    "final_output": ""
})

print("\n=== 最終狀態 ===")
for k, v in result.items():
    print(f"  {k}: {v}")

---
## 4. Pydantic Model 在 LangGraph 中的角色

Pydantic `BaseModel` 在 LangGraph 中有兩個重要用途：

1. **結構化輸出**：讓 LLM 回傳符合 schema 的 JSON
2. **Tool 定義**：定義工具的輸入參數

以下示範用 `with_structured_output()` 從使用者查詢中提取結構化資訊。

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 定義結構化輸出的 schema
class UserIntent(BaseModel):
    """從使用者查詢中提取的意圖"""
    task_type: str = Field(description="任務類型，例如：查詢、計算、翻譯、摘要")
    subject: str = Field(description="任務的主題或對象")
    urgency: str = Field(description="緊急程度：高/中/低")
    language: str = Field(description="使用者使用的語言")

# 建立 LLM，綁定結構化輸出
llm = ChatOpenAI(model="gpt-4.1", temperature=0)
structured_llm = llm.with_structured_output(UserIntent)

# 測試：從自然語言中提取結構化資訊
test_queries = [
    "請幫我趕快翻譯這段英文文章，老闆等著要",
    "Can you summarize the latest AI research paper?",
    "幫我算一下 1234 * 5678 等於多少",
]

for query in test_queries:
    result = structured_llm.invoke(f"分析以下使用者查詢：{query}")
    print(f"查詢: {query}")
    print(f"  task_type: {result.task_type}")
    print(f"  subject: {result.subject}")
    print(f"  urgency: {result.urgency}")
    print(f"  language: {result.language}")
    print()

In [ ]:
# 在 LangGraph 中使用結構化輸出
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

class IntentState(TypedDict):
    user_query: str
    intent: Optional[dict]  # 提取的意圖
    response: str

# 節點：提取意圖
def extract_intent(state: IntentState) -> dict:
    structured_llm = ChatOpenAI(model="gpt-4.1", temperature=0).with_structured_output(UserIntent)
    intent = structured_llm.invoke(f"分析以下使用者查詢：{state['user_query']}")
    print(f"[extract_intent] 提取到: {intent.model_dump()}")
    return {"intent": intent.model_dump()}

# 節點：根據意圖生成回應
def generate_response(state: IntentState) -> dict:
    intent = state["intent"]
    response = (
        f"收到！這是一個「{intent['task_type']}」任務，"
        f"主題是「{intent['subject']}」，"
        f"緊急程度：{intent['urgency']}"
    )
    print(f"[generate_response] {response}")
    return {"response": response}

# 建立 Graph
intent_graph = StateGraph(IntentState)
intent_graph.add_node("extract", extract_intent)
intent_graph.add_node("respond", generate_response)
intent_graph.add_edge(START, "extract")
intent_graph.add_edge("extract", "respond")
intent_graph.add_edge("respond", END)

app = intent_graph.compile()
result = app.invoke({"user_query": "幫我快點查一下台積電今天的股價", "intent": None, "response": ""})
print(f"\n最終回應: {result['response']}")

---
## 5. 實作 ReAct Pattern

### ReAct = Reasoning + Acting

ReAct 模式讓 Agent 在每一步：
1. **Think（思考）**：分析目前的狀態，決定下一步行動
2. **Act（行動）**：呼叫工具（Tool）執行操作
3. **Observe（觀察）**：取得工具回傳的結果
4. **重複**：直到任務完成

```
Think → Act → Observe → Think → Act → Observe → ... → 最終回答
```

這是最基礎也最常用的 Agent 模式。

In [ ]:
# 定義工具
from langchain_core.tools import tool
import ast
import operator as op

# 安全的數學運算（不使用 eval）
_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub,
    ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _safe_eval(node):
    """安全地計算數學表達式 AST"""
    if isinstance(node, ast.Constant):
        return node.value
    elif isinstance(node, ast.BinOp):
        left = _safe_eval(node.left)
        right = _safe_eval(node.right)
        return _ALLOWED_OPS[type(node.op)](left, right)
    elif isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    elif isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    else:
        raise ValueError(f"不支援的運算: {type(node)}")

@tool
def calculator(expression: str) -> str:
    """計算數學表達式。輸入一個數學表達式字串，回傳計算結果。"""
    try:
        tree = ast.parse(expression, mode='eval')
        result = _safe_eval(tree)
        return f"計算結果: {expression} = {result}"
    except Exception as e:
        return f"計算錯誤: {e}"

@tool
def search(query: str) -> str:
    """搜尋資訊。輸入搜尋關鍵字，回傳搜尋結果。"""
    # Mock 搜尋結果（教學用）
    mock_data = {
        "台積電": "台積電 (TSMC) 目前股價約 NT$850，市值全球前十大。",
        "天氣": "台北今天晴天，氣溫 28 度 C，濕度 65%。",
        "人口": "台灣人口約 2,340 萬人（2024年）。",
    }
    for key, value in mock_data.items():
        if key in query:
            return value
    return f"搜尋「{query}」：找到一些相關資訊，但無法確定具體答案。"

tools = [calculator, search]
print("已定義工具:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

In [ ]:
# 使用 create_react_agent 快速建立 ReAct Agent
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# 建立 LLM
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

# 一行建立 ReAct Agent
react_agent = create_react_agent(llm, tools)

# 測試 1：需要計算的問題
print("=== 測試 1：計算問題 ===")
result = react_agent.invoke({
    "messages": [("user", "請幫我計算 (123 + 456) * 789 等於多少？")]
})
# 印出對話過程
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, 'content') and msg.content:
        print(f"  [{role}] {msg.content[:200]}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [Tool Call] {tc['name']}({tc['args']})")

In [ ]:
# 測試 2：需要搜尋的問題
print("=== 測試 2：搜尋問題 ===")
result = react_agent.invoke({
    "messages": [("user", "台積電目前的股價是多少？")]
})
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, 'content') and msg.content:
        print(f"  [{role}] {msg.content[:200]}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [Tool Call] {tc['name']}({tc['args']})")

In [ ]:
# 測試 3：需要搜尋 + 計算的複合問題
print("=== 測試 3：複合問題（搜尋 + 計算） ===")
result = react_agent.invoke({
    "messages": [("user", "台灣人口大約多少？如果每人每天喝 2 杯咖啡，全台灣一天喝幾杯？")]
})
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, 'content') and msg.content:
        print(f"  [{role}] {msg.content[:300]}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [Tool Call] {tc['name']}({tc['args']})")

---
## 6. 實作 Plan-and-Execute Pattern

### Plan-and-Execute = 先規劃，再執行

與 ReAct 不同，Plan-and-Execute 模式：
1. **Planner（規劃者）**：先把整個任務拆成多個步驟
2. **Executor（執行者）**：依序執行每個步驟
3. **Replanner（重新規劃）**：根據執行結果決定是否需要調整計劃

```
Plan → Execute Step 1 → Execute Step 2 → ... → Replan? → 最終回答
```

適合複雜的多步驟任務。

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import operator

# --- 定義結構化輸出 ---
class Plan(BaseModel):
    """任務計劃"""
    steps: list[str] = Field(description="要執行的步驟列表")

# --- 定義狀態 ---
class PlanExecuteState(TypedDict):
    task: str                                  # 原始任務
    plan: list[str]                            # 計劃步驟
    current_step_index: int                    # 目前執行到第幾步
    results: Annotated[list[str], operator.add] # 執行結果（累加）
    final_answer: str                          # 最終回答

llm = ChatOpenAI(model="gpt-4.1", temperature=0)

# --- Planner 節點：拆解任務 ---
def planner(state: PlanExecuteState) -> dict:
    print(f"\n[Planner] 正在為任務制定計劃...")
    structured_llm = llm.with_structured_output(Plan)
    plan = structured_llm.invoke(
        f"請將以下任務拆解成 2-4 個具體步驟（每個步驟要簡短明確）：\n任務：{state['task']}"
    )
    for i, step in enumerate(plan.steps):
        print(f"  步驟 {i+1}: {step}")
    return {"plan": plan.steps, "current_step_index": 0}

# --- Executor 節點：執行當前步驟 ---
def executor(state: PlanExecuteState) -> dict:
    idx = state["current_step_index"]
    step = state["plan"][idx]
    print(f"\n[Executor] 執行步驟 {idx+1}: {step}")
    
    # 用 LLM 執行這個步驟
    context = "\n".join(state["results"]) if state["results"] else "無"
    response = llm.invoke(
        f"你正在執行一個多步驟任務。\n"
        f"原始任務：{state['task']}\n"
        f"之前步驟的結果：{context}\n"
        f"請執行以下步驟並給出結果（簡短回答）：{step}"
    )
    result_text = f"步驟 {idx+1} ({step}): {response.content[:200]}"
    print(f"  結果: {response.content[:150]}")
    return {
        "results": [result_text],
        "current_step_index": idx + 1
    }

# --- 判斷是否還有步驟要執行 ---
def should_continue(state: PlanExecuteState) -> str:
    if state["current_step_index"] >= len(state["plan"]):
        print("\n所有步驟已完成，進入總結...")
        return "summarize"
    else:
        print(f"\n繼續執行下一步 ({state['current_step_index']+1}/{len(state['plan'])})")
        return "execute"

# --- Summarizer 節點：彙整結果 ---
def summarizer(state: PlanExecuteState) -> dict:
    print("\n[Summarizer] 彙整所有步驟結果...")
    all_results = "\n".join(state["results"])
    response = llm.invoke(
        f"根據以下步驟執行結果，給出一個完整的最終回答：\n"
        f"原始任務：{state['task']}\n"
        f"步驟結果：\n{all_results}\n"
        f"請用繁體中文給出簡潔的最終回答。"
    )
    return {"final_answer": response.content}

# --- 建立 Graph ---
plan_graph = StateGraph(PlanExecuteState)
plan_graph.add_node("planner", planner)
plan_graph.add_node("executor", executor)
plan_graph.add_node("summarizer", summarizer)

plan_graph.add_edge(START, "planner")
plan_graph.add_edge("planner", "executor")
# 條件邊：執行完一步後，判斷是否繼續
plan_graph.add_conditional_edges(
    "executor",
    should_continue,
    {"execute": "executor", "summarize": "summarizer"}
)
plan_graph.add_edge("summarizer", END)

plan_app = plan_graph.compile()
print("Plan-and-Execute Graph 建立完成！")

In [ ]:
# 執行 Plan-and-Execute
result = plan_app.invoke({
    "task": "分析 Python 和 JavaScript 這兩個程式語言的優缺點，並推薦適合初學者的選擇",
    "plan": [],
    "current_step_index": 0,
    "results": [],
    "final_answer": ""
})

print("\n" + "=" * 60)
print("最終回答:")
print("=" * 60)
print(result["final_answer"])

---
## 7. 實作 Supervisor Pattern

### Supervisor = 主管調度多個專家

Supervisor 模式的架構：
- **Supervisor（主管）**：接收任務，決定要交給哪個 Worker
- **Workers（工人）**：各有專長的專家節點
- **條件路由**：Supervisor 根據任務內容動態分派

```
User → Supervisor → Worker A (研究員)
                  → Worker B (程式員)
                  → Worker C (寫手)
                  → FINISH
```

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import operator

# --- 定義 Supervisor 的路由決策 ---
class RouteDecision(BaseModel):
    """Supervisor 的路由決策"""
    next_worker: Literal["researcher", "coder", "writer", "FINISH"] = Field(
        description="下一個要呼叫的 worker，或 FINISH 表示完成"
    )
    reason: str = Field(description="選擇這個 worker 的原因")

# --- 定義狀態 ---
class SupervisorState(TypedDict):
    task: str                                       # 原始任務
    messages: Annotated[list[str], operator.add]     # 對話紀錄（累加）
    next_worker: str                                 # 下一個 worker
    final_answer: str                                # 最終回答

llm = ChatOpenAI(model="gpt-4.1", temperature=0)

# --- Supervisor 節點 ---
def supervisor(state: SupervisorState) -> dict:
    print(f"\n[Supervisor] 分析任務，決定路由...")
    
    history = "\n".join(state["messages"]) if state["messages"] else "尚無工作紀錄"
    structured_llm = llm.with_structured_output(RouteDecision)
    
    decision = structured_llm.invoke(
        f"你是一個任務主管（Supervisor），負責分派工作給專家。\n"
        f"可用的專家：\n"
        f"- researcher：負責研究、蒐集資訊、分析數據\n"
        f"- coder：負責寫程式、技術實作、debug\n"
        f"- writer：負責撰寫文章、整理報告、潤稿\n"
        f"- FINISH：所有工作都已完成\n\n"
        f"原始任務：{state['task']}\n"
        f"目前工作紀錄：\n{history}\n\n"
        f"請決定下一步該交給誰，或是已經可以 FINISH。"
    )
    
    print(f"  決定: -> {decision.next_worker}（{decision.reason}）")
    return {"next_worker": decision.next_worker}

# --- Worker 節點們 ---
def researcher(state: SupervisorState) -> dict:
    print(f"\n[Researcher] 進行研究...")
    response = llm.invoke(
        f"你是一位研究員。請針對以下任務進行研究分析，提供關鍵資訊（簡潔回答）：\n"
        f"任務：{state['task']}\n"
        f"之前的工作：{'|'.join(state['messages']) if state['messages'] else '無'}"
    )
    result = f"[Researcher] {response.content[:300]}"
    print(f"  完成: {response.content[:150]}...")
    return {"messages": [result]}

def coder(state: SupervisorState) -> dict:
    print(f"\n[Coder] 撰寫程式...")
    response = llm.invoke(
        f"你是一位程式員。請針對以下任務提供程式碼或技術方案（簡潔回答）：\n"
        f"任務：{state['task']}\n"
        f"之前的工作：{'|'.join(state['messages']) if state['messages'] else '無'}"
    )
    result = f"[Coder] {response.content[:300]}"
    print(f"  完成: {response.content[:150]}...")
    return {"messages": [result]}

def writer(state: SupervisorState) -> dict:
    print(f"\n[Writer] 撰寫內容...")
    response = llm.invoke(
        f"你是一位寫手。請根據以下資訊撰寫一段完整的回答（繁體中文）：\n"
        f"任務：{state['task']}\n"
        f"之前的工作：{'|'.join(state['messages']) if state['messages'] else '無'}"
    )
    result = f"[Writer] {response.content[:300]}"
    print(f"  完成: {response.content[:150]}...")
    return {"messages": [result]}

# --- Finish 節點 ---
def finish(state: SupervisorState) -> dict:
    print(f"\n[Finish] 彙整所有工作結果...")
    all_work = "\n".join(state["messages"])
    response = llm.invoke(
        f"請根據以下工作成果，給出最終的完整回答（繁體中文）：\n"
        f"原始任務：{state['task']}\n"
        f"工作成果：\n{all_work}"
    )
    return {"final_answer": response.content}

# --- 路由函數 ---
def route_supervisor(state: SupervisorState) -> str:
    return state["next_worker"]

# --- 建立 Graph ---
supervisor_graph = StateGraph(SupervisorState)

# 加入節點
supervisor_graph.add_node("supervisor", supervisor)
supervisor_graph.add_node("researcher", researcher)
supervisor_graph.add_node("coder", coder)
supervisor_graph.add_node("writer", writer)
supervisor_graph.add_node("finish", finish)

# 設定邊
supervisor_graph.add_edge(START, "supervisor")

# Supervisor 的條件路由
supervisor_graph.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "researcher": "researcher",
        "coder": "coder",
        "writer": "writer",
        "FINISH": "finish",
    }
)

# 每個 Worker 完成後回到 Supervisor
supervisor_graph.add_edge("researcher", "supervisor")
supervisor_graph.add_edge("coder", "supervisor")
supervisor_graph.add_edge("writer", "supervisor")
supervisor_graph.add_edge("finish", END)

supervisor_app = supervisor_graph.compile()
print("Supervisor Graph 建立完成！")

In [ ]:
# 執行 Supervisor Pattern
result = supervisor_app.invoke({
    "task": "寫一篇關於 LangGraph 的技術介紹文章，包含程式碼範例",
    "messages": [],
    "next_worker": "",
    "final_answer": ""
})

print("\n" + "=" * 60)
print("最終回答:")
print("=" * 60)
print(result["final_answer"][:800])

---
## 8. 三種模式比較與選擇指南

### 比較表

| 比較項目 | ReAct | Plan-and-Execute | Supervisor |
|---------|-------|-------------------|------------|
| **核心思路** | 逐步思考+行動 | 先規劃再執行 | 主管分派給專家 |
| **迴圈類型** | Think->Act->Observe 迴圈 | Plan->Execute 迴圈 | Supervisor->Worker 迴圈 |
| **適合場景** | 工具呼叫、問答 | 複雜多步驟任務 | 多角色協作 |
| **優點** | 簡單直覺、容易實作 | 計劃性強、可追蹤進度 | 專業分工、可擴展 |
| **缺點** | 可能迷失方向 | 規劃不好則全盤失敗 | 複雜度較高 |
| **LLM 呼叫次數** | 中（每步一次） | 高（規劃+每步） | 高（分派+每個Worker） |
| **實作複雜度** | 低 | 中 | 高 |

### 選擇指南

- **用 ReAct** 當：
  - 任務需要呼叫工具（搜尋、計算、API）
  - 任務相對簡單，不需要複雜規劃
  - 想快速建立一個有工具能力的 Agent

- **用 Plan-and-Execute** 當：
  - 任務很複雜，需要拆成多個子任務
  - 需要先思考全局再行動
  - 需要追蹤每個步驟的進度

- **用 Supervisor** 當：
  - 任務涉及多個專業領域
  - 需要不同角色的 Agent 協作
  - 需要動態決定由誰來處理哪個部分

In [ ]:
# 快速回顧：三種模式的程式碼骨架

print("""
=== 三種模式的程式碼骨架 ===

1. ReAct Pattern:
   -----------------------------------------------
   tools = [tool_a, tool_b]
   agent = create_react_agent(llm, tools)
   agent.invoke({"messages": [...]})

2. Plan-and-Execute Pattern:
   -----------------------------------------------
   graph.add_node("planner", planner)
   graph.add_node("executor", executor)
   graph.add_node("summarizer", summarizer)
   graph.add_conditional_edges("executor", should_continue, ...)

3. Supervisor Pattern:
   -----------------------------------------------
   graph.add_node("supervisor", supervisor)
   graph.add_node("worker_a", worker_a)
   graph.add_node("worker_b", worker_b)
   graph.add_conditional_edges("supervisor", route, ...)
   graph.add_edge("worker_a", "supervisor")  # Worker 回報給 Supervisor
   graph.add_edge("worker_b", "supervisor")

共同關鍵：
  - TypedDict 定義狀態
  - flat function 定義節點
  - StateGraph 串接一切
  - Conditional Edge 做動態路由
""")